# MathScy Evaluation Dashboard

Comprehensive evaluation of the MathScy pipeline:
1. **Data Coverage Analysis** - How well does our sample represent the full dataset?
2. **Extraction Quality** - Accuracy of theorem extraction and knowledge structuring
3. **MoE Expert Coverage** - Are all domain experts adequately trained?
4. **Conjecture Quality** - Novelty, difficulty, and formalizability metrics
5. **Lean 4 Verification Results** - Success rates of autoformalization
6. **End-to-End Pipeline Metrics** - Overall system performance

In [ ]:
import os, json, sys
import numpy as np
from collections import Counter, defaultdict
from pathlib import Path

BASE_DIR = '/scratch/ctoxtli/moexp'
DATA_DIR = os.path.join(BASE_DIR, 'data')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')

sys.path.insert(0, os.path.join(BASE_DIR, 'scripts'))
from data_utils import MATH_DOMAIN_GROUPS, get_domain_group, get_primary_category

print('Evaluation dashboard loaded.')

## 1. Data Coverage Analysis

In [ ]:
# Load dataset statistics
with open(os.path.join(BASE_DIR, 'dataset_stats.json')) as f:
    stats = json.load(f)

print(f"Full dataset: {stats['total_entries']:,} entries")
print(f"Categories: {len(stats['categories'])} unique")
print(f"Primary categories: {len(stats['primary_categories'])} unique")

# Compare with stratified sample
sample_path = os.path.join(DATA_DIR, 'stratified_sample_100.jsonl')
if os.path.exists(sample_path):
    sample = [json.loads(line) for line in open(sample_path)]
    sample_cats = Counter(get_primary_category(e.get('categories', '')) for e in sample)
    sample_domains = Counter(get_domain_group(e.get('categories', '')) for e in sample)
    
    print(f"\nStratified sample: {len(sample)} entries from {len(sample_cats)} categories")
    
    # Coverage analysis
    full_cats = set(k for k in stats['primary_categories'] if k.startswith('math.'))
    sample_cat_set = set(sample_cats.keys())
    print(f"\nCategory coverage: {len(sample_cat_set & full_cats)}/{len(full_cats)} math categories")
    missing = full_cats - sample_cat_set
    if missing:
        print(f"Missing categories: {missing}")
    
    print(f"\nDomain group distribution (sample):")
    for domain, count in sample_domains.most_common():
        full_count = stats.get('domain_groups', {}).get(domain, 0)
        pct = count / len(sample) * 100
        full_pct = full_count / stats['total_entries'] * 100 if stats['total_entries'] > 0 else 0
        print(f"  {domain}: {count} ({pct:.1f}% sample) vs {full_count:,} ({full_pct:.1f}% full)")

## 2. Extraction Quality Assessment

In [ ]:
# Analyze Gemini extraction results
extracted_path = os.path.join(DATA_DIR, 'extracted_knowledge.jsonl')
if os.path.exists(extracted_path):
    extracted = [json.loads(line) for line in open(extracted_path)]
    
    # Parse success rate
    parse_errors = sum(1 for e in extracted if e.get('extracted', {}).get('parse_error'))
    print(f"Gemini extraction results:")
    print(f"  Total processed: {len(extracted)}")
    print(f"  Parse errors: {parse_errors} ({parse_errors/len(extracted)*100:.1f}%)")
    print(f"  Successful: {len(extracted) - parse_errors} ({(len(extracted)-parse_errors)/len(extracted)*100:.1f}%)")
    
    # Domain coverage
    domains = Counter(e.get('domain_group', 'unknown') for e in extracted if not e.get('extracted', {}).get('parse_error'))
    print(f"\n  By domain:")
    for d, c in domains.most_common():
        print(f"    {d}: {c}")
    
    # Statement types
    types = Counter(e.get('extracted', {}).get('statement_type', 'unknown') for e in extracted if not e.get('extracted', {}).get('parse_error'))
    print(f"\n  By statement type:")
    for t, c in types.most_common():
        print(f"    {t}: {c}")
    
    # Formalization difficulty distribution
    difficulties = Counter(e.get('extracted', {}).get('lean4_formalization_difficulty', 'unknown') for e in extracted if not e.get('extracted', {}).get('parse_error'))
    print(f"\n  Lean 4 formalization difficulty:")
    for d, c in difficulties.most_common():
        print(f"    {d}: {c}")
else:
    print('No Gemini extraction results yet.')

# ArXiv theorem extraction
theorems_path = os.path.join(RESULTS_DIR, 'arxiv_extracted_theorems.jsonl')
if os.path.exists(theorems_path):
    theorems = [json.loads(line) for line in open(theorems_path)]
    print(f"\nArXiv LaTeX theorem extraction:")
    print(f"  Total theorem entries: {len(theorems)}")
    thm_types = Counter(t['type'] for t in theorems)
    print(f"  By type: {dict(thm_types.most_common())}")
    papers = len(set(t['paper_id'] for t in theorems))
    print(f"  From {papers} unique papers")
    print(f"  Avg theorems per paper: {len(theorems)/papers:.1f}")

## 3. MoE Expert Domain Coverage

In [ ]:
# Check if each MoE expert domain has sufficient training data
expert_domains = [
    'algebraic_geometry', 'combinatorics', 'number_theory',
    'analysis', 'algebra', 'geometry_topology',
    'probability_statistics', 'computational'
]

print('MoE Expert Domain Training Data Availability:')
print(f'{"Domain":<25} {"Full Dataset":>12} {"Sample":>8} {"Extracted":>10} {"Status":>10}')
print('-' * 70)

# Get sample domain counts
if os.path.exists(sample_path):
    sample_domain_counts = Counter(get_domain_group(e.get('categories', '')) for e in sample)
else:
    sample_domain_counts = Counter()

# Get extracted domain counts
if os.path.exists(extracted_path):
    extracted_domain_counts = Counter(
        e.get('domain_group', 'unknown') for e in extracted
        if not e.get('extracted', {}).get('parse_error')
    )
else:
    extracted_domain_counts = Counter()

for domain in expert_domains:
    full_count = stats.get('domain_groups', {}).get(domain, 0)
    samp_count = sample_domain_counts.get(domain, 0)
    extr_count = extracted_domain_counts.get(domain, 0)
    
    status = 'OK' if extr_count >= 5 else 'NEEDS DATA' if samp_count > 0 else 'MISSING'
    print(f'{domain:<25} {full_count:>12,} {samp_count:>8} {extr_count:>10} {status:>10}')

## 4. Conjecture Generation Results

In [ ]:
# Load conjecture results if available
conj_path = os.path.join(RESULTS_DIR, 'generated_conjectures.jsonl')
ranked_path = os.path.join(RESULTS_DIR, 'ranked_conjectures.json')

if os.path.exists(conj_path):
    conjectures = [json.loads(line) for line in open(conj_path)]
    print(f'Generated conjectures: {len(conjectures)}')
    
    strategies = Counter(c.get('strategy') for c in conjectures)
    domains = Counter(c.get('domain') for c in conjectures)
    
    print(f'\nBy strategy: {dict(strategies.most_common())}')
    print(f'By domain: {dict(domains.most_common())}')
elif os.path.exists(ranked_path):
    with open(ranked_path) as f:
        ranked = json.load(f)
    print(f'Ranked conjectures: {len(ranked)}')
else:
    print('No conjectures generated yet. Run notebook 04 after knowledge extraction.')

# STP results
for domain in expert_domains[:3]:  # Priority domains
    stp_path = os.path.join(RESULTS_DIR, f'stp_{domain}_checkpoint.json')
    if os.path.exists(stp_path):
        with open(stp_path) as f:
            stp = json.load(f)
        n_rounds = len(stp.get('rounds', []))
        n_conj = len(stp.get('all_conjectures', []))
        print(f'\nSTP for {domain}: {n_rounds} rounds, {n_conj} conjectures')

## 5. Lean 4 Verification Results

In [ ]:
lean_results_path = os.path.join(RESULTS_DIR, 'lean_verification_results.json')
lean_queue_path = os.path.join(RESULTS_DIR, 'lean_verification_queue.json')

if os.path.exists(lean_results_path):
    with open(lean_results_path) as f:
        lean_results = json.load(f)
    
    statuses = Counter(r.get('status') for r in lean_results)
    print(f'Lean 4 verification results: {len(lean_results)} conjectures')
    for s, c in statuses.most_common():
        print(f'  {s}: {c} ({c/len(lean_results)*100:.1f}%)')
elif os.path.exists(lean_queue_path):
    with open(lean_queue_path) as f:
        queue = json.load(f)
    print(f'Lean 4 verification queue: {len(queue)} conjectures pending')
    print('Run notebook 03 on GPU to verify.')
else:
    print('No Lean 4 results yet. Generate conjectures first (notebook 04), then verify (notebook 03).')

## 6. End-to-End Pipeline Summary

In [ ]:
import time

summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': {
        'total_entries': stats['total_entries'],
        'categories': len(stats['categories']),
        'domain_groups': len(stats['domain_groups']),
    },
    'sample': {
        'size': len(sample) if 'sample' in dir() else 0,
        'categories_covered': len(sample_cats) if 'sample_cats' in dir() else 0,
    },
    'extraction': {
        'gemini_processed': len(extracted) if 'extracted' in dir() else 0,
        'arxiv_theorems': len(theorems) if 'theorems' in dir() else 0,
    },
    'pipeline_status': {
        'data_preparation': 'complete',
        'knowledge_extraction': 'in_progress',
        'moe_training': 'ready_for_gpu',
        'conjecture_generation': 'ready',
        'lean4_verification': 'ready_for_gpu',
        'stp_loop': 'ready',
    },
    'gpu_requirements': {
        'phase1_moe_training': '48 GPU-hours (DGX A100 or H100)',
        'phase2_stp': '12 GPU-hours',
        'phase3_lean4': '12 GPU-hours',
    },
}

print(json.dumps(summary, indent=2))

# Save summary
with open(os.path.join(RESULTS_DIR, 'pipeline_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved to {RESULTS_DIR}/pipeline_summary.json')